# Install package

In [ ]:
%%capture

!pip install chromadb sentence-transformers rank_bm25 numpy py_vncorenlp rouge_score bert_score vncorenlp

In [ ]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi, BM25Plus
import numpy as np
from nltk.tokenize import word_tokenize
import pandas as pd
import os

# Đường dẫn tới file stopwords
# Nếu bạn có file stopwords trên Google Drive hoặc có sẵn trong Colab, hãy thay đổi đường dẫn tương ứng
filename = '/kaggle/input/bm25-stopwords/vietnamese-stopwords.txt'  # Điều chỉnh đường dẫn nếu cần

# Đọc stopwords từ file txt
with open(filename, 'r', encoding='utf-8') as f:
    list_stopwords = f.read().splitlines()

print(f"Số lượng stopwords đã tải: {len(list_stopwords)}")


In [ ]:
#Thiết lập cho pandas
pd.set_option("display.max_rows", None) 
pd.set_option("display.max_columns", None) 
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None) 

VnCoreNLP

In [ ]:
%%capture

!sudo apt-get update
!sudo apt-get install openjdk-8-jdk -y
!java -version

In [ ]:
!mkdir vncorenlp

In [ ]:
%%capture

import py_vncorenlp

# Automatically download VnCoreNLP components from the original repository
# and save them in some local working folder
py_vncorenlp.download_model(save_dir='/kaggle/working/vncorenlp',)

# Load VnCoreNLP from the local working folder that contains both `VnCoreNLP-1.2.jar` and `models` 
nlp = py_vncorenlp.VnCoreNLP(save_dir='/kaggle/working/vncorenlp')

In [ ]:
test_output = nlp.word_segment("Miễn anh văn 1, tiếng anh 2 đối với sinh viên có chứng chỉ")
print(test_output)

In [ ]:
test_output = nlp.word_segment("Miễn Anh văn 1, tiếng anh 2 đối với sinh viên có chứng chỉ")
print(test_output)

In [ ]:
test_output = nlp.word_segment("Miễn Anh văn 1, tiếng anh 2 đối với Sinh viên có chứng chỉ")
print(test_output)

In [ ]:
test_output = nlp.word_segment("Điều 15. Học bổng Sinh viên CTTT cũng được xét nhận học bổng khuyến khích học tập của Nhà nước theo...")
print(test_output)

## Main

In [ ]:
from vncorenlp import VnCoreNLP
from collections import Counter
import math
from nltk.translate.bleu_score import SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch

In [ ]:
def set_SEED():

    SEED = 42

    # random.seed(SEED)

    np.random.seed(SEED)

    torch.manual_seed(SEED)

    torch.cuda.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.enabled = False

    torch.backends.cudnn.benchmark = False

    torch.backends.cudnn.deterministic = True

    #os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
set_SEED()

In [ ]:
class StopwordRemover:
    def __init__(self, stopwords):
        self.stopwords = set(stopwords)
    
    def remove_stopwords(self, text):
        pre_text = []
        words = text.split()
        for word in words:
            if word.lower() not in self.stopwords:
                pre_text.append(word)
        return ' '.join(pre_text)


In [ ]:
class VNPreprocessor:
    def __init__(self, nlp):
        self.nlp = nlp
    
    def preprocess(self, text):
        # Sử dụng VNCoreNLP để phân đoạn từ
        segmented_sentences = self.nlp.word_segment(text)
        # segmented_sentences là một danh sách các chuỗi đã được phân đoạn từ
        # Ví dụ: ['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', ...]
        # Chúng ta sẽ nối các câu này lại thành một chuỗi duy nhất
        processed_text = ' '.join(segmented_sentences)
        return processed_text


In [ ]:
class SbertEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_name="bkai-foundation-models/vietnamese-bi-encoder"):
        self.model = SentenceTransformer(model_name)

    def __call__(self, input: Documents) -> Embeddings:
        # Tạo embeddings cho các documents
        embeddings = self.model.encode(input, convert_to_tensor=True)
        return embeddings.cpu().numpy()  # Trả về numpy array để tương thích với ChromaDB


In [ ]:
csv_filename = '/kaggle/input/rag-vietnamese-legal-text/train.csv'  # Thay đổi đường dẫn này

# Kiểm tra xem file CSV có tồn tại không
if not os.path.exists(csv_filename):
    raise FileNotFoundError(f"CSV file not found at {csv_filename}. Vui lòng kiểm tra lại đường dẫn.")

# Đọc CSV
df = pd.read_csv(csv_filename)

df = df[['context', 'document', 'article']]
df = df.drop_duplicates()

# Kiểm tra dữ liệu
print(f"Số lượng bản ghi trong dữ liệu: {len(df)}")
print(df.head())


df['context'] = df['context'].str.lower()

# Khởi tạo các đối tượng cần thiết
stopword_remover = StopwordRemover(list_stopwords)
preprocessor = VNPreprocessor(nlp)

# Tiền xử lý các context bằng VNCoreNLP
print("Tiền xử lý các context bằng VNCoreNLP...")
df['processed_context'] = df['context'].apply(preprocessor.preprocess)

# Không loại bỏ stopwords cho ChromaDB và SBERT
# Thực hiện tiền xử lý cho BM25: loại bỏ stopwords
print("Loại bỏ stopwords từ các context cho BM25...")
df['processed_context_bm25'] = df['processed_context'].apply(stopword_remover.remove_stopwords)

# Định nghĩa hàm tạo embeddings
embed_fn = SbertEmbeddingFunction()

# Khởi tạo ChromaDB client
DB_NAME = "scopus_chromadb_sbert"
chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

# Thêm documents vào cơ sở dữ liệu ChromaDB
print("Thêm documents vào ChromaDB...")
db.add(
    documents=df['processed_context'].tolist(),  # Không loại bỏ stopwords
    ids=df.index.astype(str).tolist(),  # Sử dụng chỉ số của DataFrame làm ID
    metadatas=df[['article', 'document']].to_dict(orient='records')
)

print(f"Số lượng văn bản trong ChromaDB: {db.count()}")


In [ ]:
df[['context', 'processed_context', 'processed_context_bm25']].head()

In [ ]:
# tokenized_corpus = [doc.split() for doc in df['processed_context_bm25'].tolist()]
tokenized_corpus = [doc.split() for doc in df['processed_context_bm25'].tolist()]
bm25 = BM25Plus(tokenized_corpus, k1=1.2, b=0.65) #SOTA

print("BM25 model đã được khởi tạo.")

In [ ]:
# class BM25Retriever:
#     def __init__(self, bm25_model, documents, ids, stopword_remover, df, limit=10):
#         self.bm25 = bm25_model
#         self.documents = documents
#         self.ids = ids
#         self.stopword_remover = stopword_remover
#         self.df = df
#         self.limit = limit
        
#         # Tạo từ điển ánh xạ từ doc_id sang vị trí index
#         self.id_to_pos = {doc_id: idx for idx, doc_id in enumerate(self.ids)}
        
#     def query(self, segmented_question, document_filter=None, limit=None):
#         if limit is None:
#             limit = self.limit
        
#         # Xử lý query
#         segmented_question = segmented_question.lower()
#         segmented_question = preprocessor.preprocess(segmented_question)
#         processed_query = self.stopword_remover.remove_stopwords(segmented_question)
    
#         # Tokenize bằng cách tách theo khoảng trắng
#         tokenized_query = processed_query.split()
#         scores = self.bm25.get_scores(tokenized_query)
    
#         if document_filter:
#             filtered_ids = self.df[self.df['document'] == document_filter].index.astype(str).tolist()
        
#             # Lấy tài liệu tương ứng bằng cách sử dụng id_to_pos
#             filtered_docs = [self.documents[self.id_to_pos[doc_id]] for doc_id in filtered_ids if doc_id in self.id_to_pos]
        
#             # Khởi tạo lại BM25 cho tập tài liệu đã lọc
#             bm25_filtered = BM25Okapi([doc.split() for doc in filtered_docs])
#             scores = bm25_filtered.get_scores(tokenized_query)
        
#             # Xử lý top_indices và top_ids
#             top_indices = np.argsort(scores)[::-1][:limit]
#             top_ids = [filtered_ids[i] for i in top_indices]
#         else:
#             top_indices = np.argsort(scores)[::-1][:limit]
#             top_ids = [self.ids[i] for i in top_indices]

#         # Tạo kết quả với đúng điểm số
#         results = {doc_id: scores[idx] for idx, doc_id in zip(top_indices, top_ids)}
#         return results


# class ChromaDBRetriever:
#     def __init__(self, chroma_collection):
#         self.collection = chroma_collection

#     def query(self, question, document_filter=None, topk=10):
#         question = question.lower()
#         question = preprocessor.preprocess(question)

#         if document_filter:
#             filter_dict = {"document": document_filter}
#             results = self.collection.query(
#                 query_texts=[question],
#                 n_results=topk,
#                 where=filter_dict
#             )
#         else:
#             results = self.collection.query(
#                 query_texts=[question],
#                 n_results=topk
#                 # Không truyền 'where' khi không có filter
#             )
        
#         # Chuyển đổi kết quả thành dict {id: distance}
#         doc_ids = results['ids'][0]
#         distances = results['distances'][0]
#         return {doc_id: -distance for doc_id, distance in zip(doc_ids, distances)}

# class HybridRetriever:
#     def __init__(self, biencoder, bm25, topk=10, const=20, biencoder_rate=1.5, bm25_rate=1.5):
#         self.biencoder = biencoder
#         self.bm25 = bm25
#         self.raw_topk = 50
#         self.topk = topk
#         self.const = const
#         self.biencoder_rate = biencoder_rate
#         self.bm25_rate = bm25_rate

#     def query(self, question, document_filter=None, limit = None):
#         if limit is None:
#             limit = self.topk
#         # # Tiền xử lý câu hỏi bằng VNPreprocessor
#         # preprocessed_question = preprocessor.preprocess(question)
        
#         # Truy vấn BM25 với câu hỏi đã được tiền xử lý (loại bỏ stopwords)
#         bm25_results = self.bm25.query(question, document_filter=document_filter, limit=self.raw_topk)
        
#         # Truy vấn ChromaDB với câu hỏi gốc (không loại bỏ stopwords)
#         biencoder_results = self.biencoder.query(question, document_filter=document_filter, topk=self.raw_topk)
        
#         # Chuyển kết quả thành dict {id: rank}
#         bm25_results = {k: i+1 for i, k in enumerate(bm25_results.keys())}
#         biencoder_results = {k: i+1 for i, k in enumerate(biencoder_results.keys())}

#         print('Scaled BM25 Results: ', bm25_results)
#         print('Scaled Bi-encoder Results: ', biencoder_results)
        
#         # Reciprocal Rank Fusion (RRF)
#         rrf_results = {}

#         # Tập hợp tất cả các doc_id từ cả hai kết quả
#         all_ids = set(list(bm25_results.keys()) + list(biencoder_results.keys()))
#         print('All ids context: ', all_ids)
        
#         for doc in set(list(biencoder_results.keys())):
#             print('--' * 25)
#             score = 0
            
#             score += self.biencoder_rate / (self.const + biencoder_results[doc])
            
#             if doc in bm25_results:
#                 print('Giống')
#                 print('Doc ids: ', doc)
#                 score += self.bm25_rate / (self.const + bm25_results[doc])
#             else:
#                 # score += self.bm25_rate / (self.const + limit)
#                 print('Khác')
#                 score += 0
            
#             rrf_results[doc] = score
#             print('kết quả nè: ', score)

#         # Sắp xếp các tài liệu theo điểm từ cao đến thấp
#         rrf_sorted = sorted(rrf_results.items(), key=lambda item: item[1], reverse=True)
#         print('Reciprocal Rank Fusion (RRF): ', rrf_sorted)
        
#         top_docs = [doc_id for doc_id, score in rrf_sorted[:limit]]
#         print('Top doc ids: ', top_docs)

#         return top_docs


In [ ]:
class BM25Retriever:
    def __init__(self, bm25_model, documents, ids, stopword_remover, df, limit=10):
        self.bm25 = bm25_model
        self.documents = documents
        self.ids = ids
        self.stopword_remover = stopword_remover
        self.df = df
        self.limit = limit
        
        # Tạo từ điển ánh xạ từ doc_id sang vị trí index
        self.id_to_pos = {doc_id: idx for idx, doc_id in enumerate(self.ids)}
        
    def query(self, segmented_question, document_filter=None, limit=None):
        if limit is None:
            limit = self.limit
        
        # Xử lý query
        segmented_question = segmented_question.lower()
        segmented_question = preprocessor.preprocess(segmented_question)
        processed_query = self.stopword_remover.remove_stopwords(segmented_question)
    
        # Tokenize bằng cách tách theo khoảng trắng
        tokenized_query = processed_query.split()
        scores = self.bm25.get_scores(tokenized_query)
    
        if document_filter:
            filtered_ids = self.df[self.df['document'].isin(document_filter)].index.astype(str).tolist()
        
            # Lấy tài liệu tương ứng bằng cách sử dụng id_to_pos
            filtered_docs = [self.documents[self.id_to_pos[doc_id]] for doc_id in filtered_ids if doc_id in self.id_to_pos]
        
            # Khởi tạo lại BM25 cho tập tài liệu đã lọc
            bm25_filtered = BM25Okapi([doc.split() for doc in filtered_docs])
            scores = bm25_filtered.get_scores(tokenized_query)
        
            # Xử lý top_indices và top_ids
            top_indices = np.argsort(scores)[::-1][:limit]
            top_ids = [filtered_ids[i] for i in top_indices]
        else:
            top_indices = np.argsort(scores)[::-1][:limit]
            top_ids = [self.ids[i] for i in top_indices]

        # Tạo kết quả với đúng điểm số
        results = {doc_id: scores[idx] for idx, doc_id in zip(top_indices, top_ids)}
        return results


class ChromaDBRetriever:
    def __init__(self, chroma_collection):
        self.collection = chroma_collection

    def query(self, question, document_filter=None, topk=10):
        question = question.lower()
        question = preprocessor.preprocess(question)

        if document_filter:
            filter_dict = {"document": {"$in": document_filter}}
            results = self.collection.query(
                query_texts=[question],
                n_results=topk,
                where=filter_dict
            )
        else:
            results = self.collection.query(
                query_texts=[question],
                n_results=topk
                # Không truyền 'where' khi không có filter
            )
        
        # Chuyển đổi kết quả thành dict {id: distance}
        doc_ids = results['ids'][0]
        distances = results['distances'][0]
        return {doc_id: -distance for doc_id, distance in zip(doc_ids, distances)}


class HybridRetriever:
    def __init__(self, biencoder, bm25, topk=10, alpha=0.35):
        self.biencoder = biencoder
        self.bm25 = bm25
        self.raw_topk= 50
        self.topk = topk
        self.alpha = alpha

    def scale_scores(self, scores_dict):
        """
        Scale the scores in the dictionary to the range [0, 1].

        Args:
            scores_dict (dict): Dictionary with scores to scale.

        Returns:
            dict: Dictionary with scaled scores.
        """
        values = list(scores_dict.values())
        min_val = min(values)
        max_val = max(values)
        
        if max_val == min_val:
            # Avoid division by zero; assign all scaled scores to 1.0
            return {k: 1.0 for k in scores_dict}
        else:
            return {k: (v - min_val) / (max_val - min_val) for k, v in scores_dict.items()}

    def query(self, question, document_filter=None, limit=None):
        if limit is None:
            limit = self.topk

        # Truy vấn BM25 với câu hỏi đã được tiền xử lý (loại bỏ stopwords)
        bm25_results = self.bm25.query(question, document_filter=document_filter, limit=self.raw_topk)
        
        # Truy vấn ChromaDB với câu hỏi gốc (không loại bỏ stopwords)
        biencoder_results = self.biencoder.query(question, document_filter=document_filter, topk=self.raw_topk)

        # print('Kết quả bm25: ', bm25_results)
        # print('Kết quả biencoder: ', biencoder_results)
        
        # Scale the scores to [0, 1]
        bm25_scaled = self.scale_scores(bm25_results)
        biencoder_scaled = self.scale_scores(biencoder_results)

        # print('Scaled BM25 Results: ', bm25_scaled)
        # print('Scaled Bi-encoder Results: ', biencoder_scaled)
        
        # Reciprocal Rank Fusion (RRF) may not directly utilize scaled scores.
        # If you intend to use scaled scores for fusion, consider using a weighted sum approach instead.
        # Below is an example of weighted sum fusion using scaled scores.

        fused_scores = {}

        # Get all unique document IDs from both retrievers
        all_ids = set(bm25_scaled.keys()).union(set(biencoder_scaled.keys()))
        # print('All document IDs:', all_ids)

        for doc_id in all_ids:
            bm25_score = bm25_scaled.get(doc_id, 0)  # Default to 0 if not present
            biencoder_score = biencoder_scaled.get(doc_id, 0)  # Default to 0 if not present

            # Weighted sum of scaled scores
            fused_score = ((1-self.alpha) * bm25_score) + (self.alpha * biencoder_score)
            # fused_score = bm25_score * biencoder_score
            fused_scores[doc_id] = fused_score

            # print(f'Doc ID: {doc_id}, BM25 Scaled: {bm25_score}, Bi-encoder Scaled: {biencoder_score}, Fused Score: {fused_score}')

        # Sort the fused scores in descending order
        fused_sorted = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)
        top_docs = [doc_id for doc_id, score in fused_sorted[:limit]]

        # print('Top Documents:', top_docs)

        return top_docs



In [ ]:
# Khởi tạo BM25Retriever
bm25_retriever = BM25Retriever(
    bm25_model=bm25,
    documents=df['processed_context_bm25'].tolist(),
    ids=df.index.astype(str).tolist(),
    stopword_remover=stopword_remover,
    df=df,
    limit=10
)

# Khởi tạo ChromaDBRetriever
chroma_retriever = ChromaDBRetriever(db)

# Khởi tạo HybridRetriever
hybrid_retriever = HybridRetriever(
    biencoder=chroma_retriever, 
    bm25=bm25_retriever, 
    topk=10
)

print("Các retriever đã được khởi tạo thành công!")


In [ ]:
# # Hàm để thực hiện truy vấn
# def perform_query(question, document_filter=None, limit=5):
#     top_documents = hybrid_retriever.query(question, document_filter=document_filter, limit=limit)
    
#     if not top_documents:
#         print("Không tìm thấy kết quả nào.")
#         return
    
#     print(f"Kết quả truy vấn cho câu hỏi: '{question}' với document filter: '{document_filter}'\n")
#     for idx, doc_id in enumerate(top_documents, 1):
#         context = df.loc[int(doc_id), 'context']
#         article = df.loc[int(doc_id), 'article']
#         document = df.loc[int(doc_id), 'document']
#         print(f"Rank{idx}")
#         print(f"{idx}. ID: {doc_id}, Article: {article}, Document: {document}")
#         print(f"   Context: {context[:200]}...")  # Hiển thị 200 ký tự đầu của context
#         print(f"-------------------------------------------------------------\n")
#     print(f"==========================================================================\n")
# # Ví dụ truy vấn
# question = "Đối với các cơ quan quản lý văn bằng, chứng chỉ sẽ làm nhiệm vụ gì khi có thanh tra?"
# document_filter = "QUY CHẾ Văn bằng, chứng chỉ của Trường Đại học Công nghệ Thông tin"  # Thay đổi thành giá trị document bạn muốn lọc

# perform_query(question, document_filter=document_filter, limit=10)

## Truy xuất với vidu

In [ ]:
# def perform_query(question, document_filter=None, gt=None):
#     top_documents = hybrid_retriever.query(question, document_filter=document_filter)
    
#     if not top_documents:
#         print("Không tìm thấy kết quả nào.")
#         return
    
#     print(f"Kết quả truy vấn cho câu hỏi: '{question}' với document filter: '{document_filter}'\n")
#     print(f'Ground truth: {gt}')
#     for idx, doc_id in enumerate(top_documents, 1):
#         # Chuyển đổi doc_id thành số nguyên nếu cần
#         doc_index = int(doc_id)
#         if doc_index in df.index:
#             context = df.loc[doc_index, 'context']
#             article = df.loc[doc_index, 'article']
#             document = df.loc[doc_index, 'document']
#             print(f"Rank {idx}")
#             print(f"{idx}. ID: {doc_id}, Article: {article}, Document: {document}")
#             print(f"   Context: {context[:200]}...")  # Hiển thị 200 ký tự đầu của context
#             print(f"-------------------------------------------------------------\n")
#         else:
#             print(f"Doc ID {doc_id} không tồn tại trong DataFrame.")
#     print(f"==========================================================================\n")

In [ ]:
# data = pd.read_csv('/kaggle/input/rag-vietnamese-legal-text/train.csv')

# for i in range(100, 200):
#     df1 = data.iloc[[i]]

#     question = df1['question'].iloc[0]
#     document_filter = df1['document'].iloc[0] # Thay đổi thành giá trị document bạn muốn lọc

#     gt = df1['article'].iloc[0]
#     perform_query(question, document_filter=document_filter, gt = gt)

## Model classify

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import pickle

In [ ]:
df_train = pd.read_csv('/kaggle/input/rag-vietnamese-legal-text/train.csv')

#Tiền xử lý
df_train['processed_question'] = df_train['question'].str.lower()
df_train['processed_question'] = df_train['processed_question'].apply(preprocessor.preprocess)
df_train['processed_question'] = df_train['processed_question'].apply(stopword_remover.remove_stopwords)

In [ ]:
df_val = pd.read_csv('/kaggle/input/rag-vietnamese-legal-text/val.csv')

#Tiền xử lý
df_val['processed_question'] = df_val['question'].str.lower()
df_val['processed_question'] = df_val['processed_question'].apply(preprocessor.preprocess)
df_val['processed_question'] = df_val['processed_question'].apply(stopword_remover.remove_stopwords)

In [ ]:
df_test = pd.read_csv('/kaggle/input/rag-vietnamese-legal-text/test.csv')

#Tiền xử lý
df_test['processed_question'] = df_test['question'].str.lower()
df_test['processed_question'] = df_test['processed_question'].apply(preprocessor.preprocess)
df_test['processed_question'] = df_test['processed_question'].apply(stopword_remover.remove_stopwords)

In [ ]:
# Mã hóa nhãn cho tài liệu
label_encoder = LabelEncoder()
df_train['document_encoded'] = label_encoder.fit_transform(df_train['document'])
df_val['document_encoded'] = label_encoder.transform(df_val['document'])
df_test['document_encoded'] = label_encoder.transform(df_test['document'])

# Chia dữ liệu thành input và label
X_train = df_train['processed_question']
X_val = df_val['processed_question']
X_test = df_test['processed_question']
y_train = df_train['document_encoded']
y_val = df_val['document_encoded']
y_test = df_test['document_encoded']

print("Data preprocessing completed!")

In [ ]:
%%capture
# 1. Cài Đặt Thư Viện
!pip install transformers datasets vncorenlp torch scikit-learn evaluate

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_lengths(df, set_name):
    # Tính độ dài dựa trên số lượng từ
    lengths = df['processed_question'].apply(lambda x: len(x.split()))
    
    # Hiển thị các thống kê cơ bản
    print(f"=== Thống Kê Độ Dài cho {set_name} ===")
    print(f"Số lượng mẫu: {len(lengths)}")
    print(f"Độ dài tối thiểu: {lengths.min()}")
    print(f"Độ dài tối đa: {lengths.max()}")
    print(f"Độ dài trung bình: {lengths.mean():.2f}")
    print(f"Độ dài trung vị (Median): {lengths.median()}")
    print(f"25th percentile: {lengths.quantile(0.25)}")
    print(f"75th percentile: {lengths.quantile(0.75)}")
    print(f"90th percentile: {lengths.quantile(0.90)}")
    print(f"95th percentile: {lengths.quantile(0.95)}")
    print(f"99th percentile: {lengths.quantile(0.99)}")
    print("\n")
    
    # Vẽ biểu đồ phân phối độ dài
    plt.figure(figsize=(10, 6))
    sns.histplot(lengths, bins=50, kde=True, color='skyblue')
    plt.title(f'Phân Phối Độ Dài Câu Hỏi trong {set_name}')
    plt.xlabel('Số Lượng Từ')
    plt.ylabel('Tần Suất')
    plt.axvline(lengths.quantile(0.95), color='red', linestyle='dashed', linewidth=1, label='95th Percentile')
    plt.legend()
    plt.show()

analyze_lengths(df_train, 'Train Set')

analyze_lengths(df_val, 'Validation Set')

analyze_lengths(df_test, 'Test Set')


In [ ]:
# 2. Import Thư Viện
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings
import torch

# 3. Tắt Cảnh Báo Không Quan Trọng
warnings.filterwarnings("ignore", category=FutureWarning)

# 4. Chuẩn Bị Dữ Liệu
# Giả sử df_train, df_val và df_test đã được định nghĩa và chứa các cột 'processed_question' và 'document'
label_encoder = LabelEncoder()
df_train['labels'] = label_encoder.fit_transform(df_train['document'])
df_val['labels'] = label_encoder.transform(df_val['document'])
df_test['labels'] = label_encoder.transform(df_test['document'])

# Kiểm tra xem có thiếu giá trị không
assert not df_train['processed_question'].isnull().any(), "Có giá trị thiếu trong 'processed_question' của df_train!"
assert not df_val['processed_question'].isnull().any(), "Có giá trị thiếu trong 'processed_question' của df_val!"
assert not df_test['processed_question'].isnull().any(), "Có giá trị thiếu trong 'processed_question' của df_test!"

# Chuyển đổi DataFrame thành Hugging Face Dataset
train_dataset = Dataset.from_pandas(df_train[['processed_question', 'labels']])
val_dataset = Dataset.from_pandas(df_val[['processed_question', 'labels']])
test_dataset = Dataset.from_pandas(df_test[['processed_question', 'labels']])

print("Data preprocessing completed!")

# 5. Tải Tokenizer từ PhoBERT với Các Tham Số Đã Được Đặt Rõ Ràng
tokenizer = AutoTokenizer.from_pretrained(
    "vinai/phobert-large",  # Sử dụng cùng một phiên bản với mô hình
    use_fast=True,  # Sử dụng fast tokenizer để tăng tốc độ
    clean_up_tokenization_spaces=True
)

# 6. Định Nghĩa Hàm Tiền Xử Lý Dữ Liệu với max_length Được Xác Định và loại bỏ token_type_ids
def preprocess_function(examples):
    return tokenizer(
        examples["processed_question"],
        padding=False,  # Sử dụng Data Collator để padding động
        truncation=True,
        max_length=128,  # Điều chỉnh giá trị này tùy thuộc vào độ dài câu hỏi
        return_token_type_ids=False  # Loại bỏ token_type_ids vì không cần thiết
    )

# 7. Tokenize Dữ Liệu với Parallel Processing
train_dataset = train_dataset.map(preprocess_function, batched=True, num_proc=1)
val_dataset = val_dataset.map(preprocess_function, batched=True, num_proc=1)
test_dataset = test_dataset.map(preprocess_function, batched=True, num_proc=1)

print("Tokenization completed!")

# 8. Sử Dụng Data Collator với Padding Động
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 9. Định Dạng Dataset để Tương Thích với PyTorch và Loại Bỏ token_type_ids nếu còn
train_dataset = train_dataset.remove_columns(["processed_question"])
val_dataset = val_dataset.remove_columns(["processed_question"])
test_dataset = test_dataset.remove_columns(["processed_question"])

# Định dạng lại Dataset để tương thích với PyTorch
train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

# 10. Tải Mô Hình PhoBERT cho Phân Loại Sequence
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-large",  # Sử dụng cùng một phiên bản với tokenizer
    num_labels=len(label_encoder.classes_),
    problem_type="single_label_classification"  # Định rõ loại vấn đề đúng cách
)

# 11. Định Nghĩa TrainingArguments với Các Tham Số Đã Được Cập Nhật và Tối Ưu Hóa Hiệu Suất
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # Đánh giá sau mỗi epoch
    save_strategy="epoch",  # Lưu mô hình sau mỗi epoch
    learning_rate=1e-5,
    per_device_train_batch_size=32,  # Tăng batch size nếu GPU cho phép
    per_device_eval_batch_size=32,
    num_train_epochs=30,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    lr_scheduler_type='linear',
    report_to="none",  # Tắt Weights & Biases nếu không sử dụng
    dataloader_num_workers=4,  # Số lượng worker cho DataLoader
    run_name="phoBERT_classification"  # Đặt tên run nếu cần
)

# 12. Định Nghĩa Hàm Tính Toán Các Chỉ Số Đánh Giá
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Chuyển logits thành tensor nếu chưa phải tensor
    logits = torch.tensor(logits)
    predictions = torch.argmax(logits, dim=-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# 13. Khởi Tạo Trainer với Các Tham Số và Dữ Liệu Đã Chuẩn Bị
early_stopping = EarlyStoppingCallback(early_stopping_patience=5)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

# 14. Huấn Luyện Mô Hình
trainer.train()

# 15. Đánh Giá Mô Hình trên Tập Validation và Test
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

# 16. Dự Đoán trên Tập Test và Đánh Giá Hiệu Suất
predictions, labels, _ = trainer.predict(test_dataset)
predicted_labels = predictions.argmax(axis=-1)

# Tính các chỉ số đánh giá chi tiết
precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_labels, average="weighted")
acc = accuracy_score(labels, predicted_labels)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1-score: {f1:.4f}")

# 17. Lưu Mô Hình và Tokenizer (Tuỳ Chọn)
trainer.save_model("./best_phobert_model")
tokenizer.save_pretrained("./best_phobert_model")


In [ ]:
# Vector hóa dữ liệu bằng TF-IDF
tfidf = TfidfVectorizer(max_features=5000)  # Giới hạn số lượng từ tối đa
X_train_encoder = tfidf.fit_transform(X_train)
X_test_encoder = tfidf.transform(X_test)

print("TF-IDF Vectorization completed!")

In [ ]:
# Huấn luyện mô hình SVM
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_encoder, y_train)

# Lưu mô hình và vectorizer
with open('/kaggle/working/svm_model.pkl', 'wb') as model_file:
    pickle.dump(svm_model, model_file)
with open('/kaggle/working/tfidf_vectorizer.pkl', 'wb') as vectorizer_file:
    pickle.dump(tfidf, vectorizer_file)
with open('/kaggle/working/label_encoder.pkl', 'wb') as encoder_file:
    pickle.dump(label_encoder, encoder_file)

print("Model and vectorizer saved successfully!")

In [ ]:
# Đánh giá mô hình
y_pred_svm = svm_model.predict(X_test_encoder)
print("SVM Results on Test Data:")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))

In [ ]:
class PredictTop5:
    def __init__(self, model, vectorizer, label_encoder):
        """
        Khởi tạo lớp dự đoán top 5 nhãn
        :param model: Mô hình đã huấn luyện (SVM)
        :param vectorizer: Bộ vectorizer (TfidfVectorizer)
        :param label_encoder: Bộ mã hóa nhãn (LabelEncoder)
        """
        self.model = model
        self.vectorizer = vectorizer
        self.label_encoder = label_encoder

    def predict(self, input_text):
        """
        Dự đoán top 5 nhãn có xác suất cao nhất
        :param input_text: Văn bản cần dự đoán
        :return: Danh sách top 5 nhãn gốc (chưa encode)
        """
        # Biến đổi văn bản thành vector
        encoded_text = self.vectorizer.transform([input_text])
        # Tính toán xác suất cho mỗi nhãn
        probabilities = self.model.predict_proba(encoded_text)[0]
        # Lấy top 5 nhãn dự đoán (theo chỉ số xác suất)
        top_5_indices = np.argsort(probabilities)[-15:][::-1]
        
        # Chuyển đổi chỉ số nhãn từ mã hóa sang nhãn gốc
        top_5_labels = self.label_encoder.inverse_transform(top_5_indices)
        
        return top_5_labels.tolist()

In [ ]:
# Load lại mô hình, vectorizer và label encoder
with open('/kaggle/working/svm_model.pkl', 'rb') as model_file:
    loaded_model = pickle.load(model_file)
with open('/kaggle/working/tfidf_vectorizer.pkl', 'rb') as vectorizer_file:
    loaded_vectorizer = pickle.load(vectorizer_file)
with open('/kaggle/working/label_encoder.pkl', 'rb') as encoder_file:
    loaded_label_encoder = pickle.load(encoder_file)

# Tạo đối tượng dự đoán
predictor = PredictTop5(loaded_model, loaded_vectorizer, loaded_label_encoder)

# Văn bản cần dự đoán
input_text = "This is a sample question to predict top 5 labels."

# Dự đoán top 5 nhãn gốc
top_5_predictions = predictor.predict(input_text)

# In kết quả
print("\nTop 5 predicted labels:")
print(top_5_predictions)

## Đánh giá truy xuất trên tập test

In [ ]:
import pandas as pd

# Đường dẫn tới tập test
test_csv_filename = '/kaggle/input/rag-vietnamese-legal-text/test.csv'  # Thay đổi đường dẫn nếu cần

# Đọc tập test
test_data = pd.read_csv(test_csv_filename)

# Kiểm tra dữ liệu
print(f"Số lượng mẫu trong tập test: {len(test_data)}")
# print(test_data.head())


In [ ]:
# test_data = test_data.sample(2)

In [ ]:
test_data['context'] = test_data['context'].str.lower()
test_data['processed_context'] = test_data['context'].apply(preprocessor.preprocess)

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Khởi tạo mô hình SBERT
model_name = "bkai-foundation-models/vietnamese-bi-encoder"  # Đảm bảo mô hình này đã được tải về
sbert_model = SentenceTransformer(model_name)


In [ ]:
def evaluate_rag_system(test_df, retriever, sbert_model, topk_list=[1, 5, 10, 15, 20, 25, 30]):
    """
    Đánh giá hệ thống RAG trên tập test bằng cosine similarity.
    
    Args:
        test_df (pd.DataFrame): Tập dữ liệu test với cột 'question' và 'ground_truth'.
        retriever (HybridRetriever): Bộ retriever đã được khởi tạo.
        sbert_model (SentenceTransformer): Mô hình SBERT để tạo embeddings.
        topk (int): Số lượng tài liệu được truy xuất.
    
    Returns:
        float: Điểm đánh giá trung bình trên toàn bộ tập test.
    """
    
    similarity_scores_dict = {k: [] for k in topk_list}
    
    for idx, row in test_df.iterrows():
        question = row['question']
        ground_truth = row['processed_context']
        gt_doc = row['document']
        #classify document
        question_2 = question.lower()
        question_2 = preprocessor.preprocess(question_2)
        question_2 = stopword_remover.remove_stopwords(question_2)

        doc_filter_list = predictor.predict(question_2)

        limit = max(topk_list)
        
        # Truy vấn hệ thống RAG để lấy topk context
        retrieved_doc_ids = retriever.query(question, document_filter=doc_filter_list, limit = limit)  # Bạn có thể thêm filter nếu cần
        retrieved_contexts = [db.get(doc_id)['documents'][0] for doc_id in retrieved_doc_ids if db.get(doc_id)]
                
        if not retrieved_contexts:
            print(f"Không tìm thấy context nào cho câu hỏi: '{question}'")
            for k in topk_list:
                similarity_scores_dict[k].append(0)
            continue
        
        # Tạo embeddings cho ground truth và các context được truy xuất
        ground_truth_embedding = sbert_model.encode(ground_truth, convert_to_tensor=True)
        retrieved_embeddings = sbert_model.encode(retrieved_contexts, convert_to_tensor=True)
        
        # Tính cosine similarity giữa ground truth và từng context được truy xuất
        cosine_scores = util.cos_sim(ground_truth_embedding, retrieved_embeddings)[0].cpu().numpy()
        
        # Lấy giá trị cosine similarity cao nhất
        for k in topk_list:
            topk_cos_scores = cosine_scores[:k]
            max_similarity = np.max(topk_cos_scores) if topk_cos_scores.size > 0 else 0
            similarity_scores_dict[k].append(max_similarity if max_similarity >= 0.95 else 0)

        result_print = False
        
        if result_print:
        # In thông tin mỗi bước nếu cần
            print(f"Sample {idx}:")
            print(f"Question: {question}")
            print(f"Classify: {doc_filter_list}")
            print(f"Ground Truth_doc: {gt_doc}")
            print(f"Ground Truth: {ground_truth}")
            
            print("-------------------------------------------------------------")
            for rank, (doc_id, sim) in enumerate(zip(retrieved_doc_ids, cosine_scores), 1):
                try:
                    context = db.get(doc_id)['documents'][0] # documents: context in db
                    print(f"Context retrieved: {context}")
                except (KeyError, ValueError):
                    context = "Not found"
                    print(f"Context retrieved: {context}")
                print(f"Rank {rank}: Doc ID {doc_id}, Cosine Similarity: {sim:.4f}")
            print(f"Max Cosine Similarity: {max_similarity:.4f}")
            print("=======================================================================\n")
    
    # Tính điểm đánh giá trung bình
    return {k: np.mean(v) for k, v in similarity_scores_dict.items()}


In [ ]:
list_k = [1, 5, 10, 15, 20, 25, 30]

In [ ]:
%%capture
# Đảm bảo rằng hệ thống retriever đã được khởi tạo
# Giả sử bạn đã khởi tạo hybrid_retriever như trong mã nguồn trước đó

# Đánh giá hệ thống trên tập test
average_similarity = evaluate_rag_system(
    test_df=test_data,
    retriever=hybrid_retriever,
    sbert_model=sbert_model,
    topk_list = list_k
)

In [ ]:
for k in list_k:
    print(f"Average score for topk={k}: {average_similarity[k]:.4f}")

**Hybrid Retriever: BiEncoder(VoVanPhuc) + BM25Okapi(k1 = 1.2, b = 6.5), weight ensemble = 0.2**

 - Average score for topk=1: 0.6615

 - Average score for topk=5: 0.8965

 - Average score for topk=10: 0.9406

 - Average score for topk=15: 0.9621

 - Average score for topk=20: 0.9713

 - Average score for topk=25: 0.9734

 - Average score for topk=30: 0.9775


**Hybrid Retriever: BiEncoder(VoVanPhuc) + BM25Okapi(k1 = 1.2, b = 6.5), weight ensemble = 0.7**

 - Average score for topk=1: 0.6308

 - Average score for topk=5: 0.8770

 - Average score for topk=10: 0.9354

 - Average score for topk=15: 0.9559

 - Average score for topk=20: 0.9621

 - Average score for topk=25: 0.9682

 - Average score for topk=30: 0.9693


**Hybrid Retriever: BiEncoder(VoVanPhuc) + BM25Okapi(k1 = 1.2, b = 6.5), weight ensemble = 0.35**

 - Average score for topk=1: 0.6553

 - Average score for topk=5: 0.8985

 - Average score for topk=10: 0.9477

 - Average score for topk=15: 0.9621

 - Average score for topk=20: 0.9682

 - Average score for topk=25: 0.9693

 - Average score for topk=30: 0.9754


### SOTA - BKai
**Hybrid Retriever: BiEncoder(BKai) + BM25Okapi(k1 = 1.2, b = 6.5), weight ensemble = 0.35**

- Average score for topk=1: 0.6926

- Average score for topk=5: 0.9211

- Average score for topk=10: 0.9570

- Average score for topk=15: 0.9662

- Average score for topk=20: 0.9723

- Average score for topk=25: 0.9744

- Average score for topk=30: 0.9785


## Truy xuất với nhiều document filter

In [ ]:
class BM25Retriever:
    def __init__(self, bm25_model, documents, ids, stopword_remover, df, limit=10):
        self.bm25 = bm25_model
        self.documents = documents
        self.ids = ids
        self.stopword_remover = stopword_remover
        self.df = df
        self.limit = limit
        
        # Tạo từ điển ánh xạ từ doc_id sang vị trí index
        self.id_to_pos = {doc_id: idx for idx, doc_id in enumerate(self.ids)}
        
    def query(self, segmented_question, document_filters=None, limit=None):
        if limit is None:
            limit = self.limit
        
        # Xử lý query
        segmented_question = segmented_question.lower()
        segmented_question = preprocessor.preprocess(segmented_question)
        processed_query = self.stopword_remover.remove_stopwords(segmented_question)
    
        # Tokenize bằng cách tách theo khoảng trắng
        tokenized_query = processed_query.split()
        scores = self.bm25.get_scores(tokenized_query)
    
        if document_filters:
            if isinstance(document_filters, str):
                document_filters = [document_filters]
            
            filtered_ids = self.df[self.df['document'].isin(document_filters)].index.astype(str).tolist()
        
            # Lấy tài liệu tương ứng bằng cách sử dụng id_to_pos
            filtered_docs = [self.documents[self.id_to_pos[doc_id]] for doc_id in filtered_ids if doc_id in self.id_to_pos]
        
            # Khởi tạo lại BM25 cho tập tài liệu đã lọc
            bm25_filtered = BM25Okapi([doc.split() for doc in filtered_docs])
            scores = bm25_filtered.get_scores(tokenized_query)
        
            # Xử lý top_indices và top_ids
            top_indices = np.argsort(scores)[::-1][:limit]
            top_ids = [filtered_ids[i] for i in top_indices]
        else:
            top_indices = np.argsort(scores)[::-1][:limit]
            top_ids = [self.ids[i] for i in top_indices]

        # Tạo kết quả với đúng điểm số
        results = {doc_id: scores[idx] for idx, doc_id in zip(top_indices, top_ids)}
        return results


In [ ]:
class ChromaDBRetriever:
    def __init__(self, chroma_collection):
        self.collection = chroma_collection

    def query(self, question, document_filters=None, topk=10):
        question = question.lower()
        question = preprocessor.preprocess(question)

        if document_filters:
            if isinstance(document_filters, str):
                document_filters = [document_filters]
            
            # Tạo điều kiện lọc với nhiều tài liệu
            filter_dict = {"document": {"$in": document_filters}}
            results = self.collection.query(
                query_texts=[question],
                n_results=topk,
                where=filter_dict
            )
        else:
            results = self.collection.query(
                query_texts=[question],
                n_results=topk
                # Không truyền 'where' khi không có filter
            )
        
        # Chuyển đổi kết quả thành dict {id: distance}
        doc_ids = results['ids'][0]
        distances = results['distances'][0]
        return {doc_id: -distance for doc_id, distance in zip(doc_ids, distances)}


In [ ]:
class HybridRetriever:
    def __init__(self, biencoder, bm25, topk=10, alpha=0.35):
        self.biencoder = biencoder
        self.bm25 = bm25
        self.raw_topk= 50
        self.topk = topk
        self.alpha = alpha

    def scale_scores(self, scores_dict):
        """
        Scale the scores in the dictionary to the range [0, 1].

        Args:
            scores_dict (dict): Dictionary with scores to scale.

        Returns:
            dict: Dictionary with scaled scores.
        """
        values = list(scores_dict.values())
        min_val = min(values)
        max_val = max(values)
        
        if max_val == min_val:
            # Avoid division by zero; assign all scaled scores to 1.0
            return {k: 1.0 for k in scores_dict}
        else:
            return {k: (v - min_val) / (max_val - min_val) for k, v in scores_dict.items()}

    def query(self, question, document_filters=None, limit=None):
        if limit is None:
            limit = self.topk

        # Truy vấn BM25 với câu hỏi đã được tiền xử lý (loại bỏ stopwords)
        bm25_results = self.bm25.query(question, document_filters=document_filters, limit=self.raw_topk)
        
        # Truy vấn ChromaDB với câu hỏi gốc (không loại bỏ stopwords)
        biencoder_results = self.biencoder.query(question, document_filters=document_filters, topk=self.raw_topk)

        # Scale the scores to [0, 1]
        bm25_scaled = self.scale_scores(bm25_results)
        biencoder_scaled = self.scale_scores(biencoder_results)

        fused_scores = {}

        # Get all unique document IDs from both retrievers
        all_ids = set(bm25_scaled.keys()).union(set(biencoder_scaled.keys()))

        for doc_id in all_ids:
            bm25_score = bm25_scaled.get(doc_id, 0)  # Default to 0 if not present
            biencoder_score = biencoder_scaled.get(doc_id, 0)  # Default to 0 if not present

            # Weighted sum of scaled scores
            fused_score = ((1 - self.alpha) * bm25_score) + (self.alpha * biencoder_score)
            fused_scores[doc_id] = fused_score

        # Sort the fused scores in descending order
        fused_sorted = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)
        top_docs = [doc_id for doc_id, score in fused_sorted[:limit]]

        return top_docs
